## Cell 1: setup, load and combine

In [1]:
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

BASE = os.path.join(os.path.expanduser('~'), 'fraud-outputs')
TBL_DIR = os.path.join(BASE, 'tables')
os.makedirs(TBL_DIR, exist_ok=True)

train = pd.read_csv('data/raw/fraudTrain.csv', parse_dates=['trans_date_trans_time', 'dob'])
test = pd.read_csv('data/raw/fraudTest.csv', parse_dates=['trans_date_trans_time', 'dob'])
train['split'] = 'train'
test['split'] = 'test'

df = pd.concat([train, test], ignore_index=True)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df['merchant'] = df['merchant'].str.replace('^fraud_', '', regex=True)
df = df.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

print(df['split'].value_counts())
print(df.shape)

split
train    1296675
test      555719
Name: count, dtype: int64
(1852394, 23)


## Cell 2: time, customer and location features

In [2]:
t = df['trans_date_trans_time']
df['hour'] = t.dt.hour
df['is_night'] = df['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
df['day_of_week'] = t.dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['age'] = ((t - df['dob']).dt.days / 365.25).astype(int)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine_km(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df['log_amt'] = np.log1p(df['amt'])
df[['hour', 'is_night', 'is_weekend', 'age', 'distance_km', 'log_amt']].describe().T

,count,mean,std,min,25%,50%,75%,max
hour,1852394.0,12.806119,6.815753,0.000000,7.000000,14.000000,19.000000,23.000000
is_night,1852394.0,0.234843,0.423901,0.000000,0.000000,0.000000,0.000000,1.000000
is_weekend,1852394.0,0.327632,0.469350,0.000000,0.000000,0.000000,1.000000,1.000000
age,1852394.0,45.763933,17.412384,13.000000,32.000000,44.000000,57.000000,96.000000
distance_km,1852394.0,76.111726,29.116970,0.022255,55.320087,78.216380,98.509467,152.117173
log_amt,1852394.0,3.531722,1.288350,0.693147,2.364620,3.880532,4.432007,10.273322


## Cell 3: behaviour features (past only)

In [3]:
g = df.groupby('cc_num')

# Hours since the card's previous transaction
df['hours_since_prev'] = g['trans_date_trans_time'].diff().dt.total_seconds() / 3600
df['is_first_txn'] = df['hours_since_prev'].isna().astype(int)
df['hours_since_prev'] = df['hours_since_prev'].fillna(720).clip(upper=720)

# Amount compared with the card's average over EARLIER transactions only
past_n = g.cumcount()
past_sum = g['amt'].cumsum() - df['amt']
df['card_avg_past_amt'] = past_sum / past_n.replace(0, np.nan)
df['amt_vs_card_avg'] = (df['amt'] / df['card_avg_past_amt']).fillna(1.0)

# Number of earlier transactions on the same card in the last 1 hour / 24 hours
card_code = pd.factorize(df['cc_num'], sort=True)[0].astype('int64')
secs = ((df['trans_date_trans_time'] - df['trans_date_trans_time'].min())
        .dt.total_seconds()).astype('int64').to_numpy()
BIG = int(secs.max()) + 10**7
keys = card_code * BIG + secs

for seconds, name in [(3600, 'txn_count_1h'), (86400, 'txn_count_24h')]:
    hi = np.searchsorted(keys, keys, side='left')
    lo = np.searchsorted(keys, keys - seconds, side='left')
    df[name] = hi - lo

df[['hours_since_prev', 'amt_vs_card_avg', 'txn_count_1h', 'txn_count_24h']].describe().T

,count,mean,std,min,25%,50%,75%,max
hours_since_prev,1852394.0,8.977963,20.766565,0.000000,1.583611,4.363889,10.609444,720.000000
amt_vs_card_avg,1852394.0,1.013238,2.591848,0.002176,0.156205,0.668358,1.197644,676.388128
txn_count_1h,1852394.0,0.199999,0.475632,0.000000,0.000000,0.000000,0.000000,8.000000
txn_count_24h,1852394.0,4.110729,3.255181,0.000000,2.000000,3.000000,6.000000,35.000000


## Cell 4: compare fraud vs legitimate, then save

In [ ]:
features = ['hour', 'is_night', 'is_weekend', 'age', 'distance_km', 'log_amt',
            'hours_since_prev', 'amt_vs_card_avg', 'txn_count_1h', 'txn_count_24h']

comp = df[df['split'] == 'train'].groupby('is_fraud')[features].median().T
comp.columns = ['Legitimate (median)', 'Fraud (median)']
comp.to_csv(os.path.join(TBL_DIR, 'fe_01_feature_comparison.csv'))

df.to_pickle(os.path.join(BASE, 'features.pkl'))
print('Saved', df.shape, 'to', BASE)
comp3

Saved (1852394, 36) to C:\Users\adith\fraud-outputs


,Legitimate (median),Fraud (median)
hour,14.000000,22.000000
is_night,0.000000,1.000000
is_weekend,0.000000,0.000000
age,43.000000,47.000000
distance_km,78.233012,77.931954
log_amt,3.877017,5.985203
hours_since_prev,4.623611,1.403889
amt_vs_card_avg,0.664867,5.163651
txn_count_1h,0.000000,0.000000
txn_count_24h,3.000000,4.000000
